In [ ]:
SESSION_UUID = "REPLACE_WITH_SESSION_UUID"

import pathlib
WORKSPACES = pathlib.Path("/home/ubuntu/lab/workspaces")
SESSION_DIR = WORKSPACES / SESSION_UUID / f"session_{SESSION_UUID}"
assert SESSION_DIR.exists(), f"Workspace not found: {SESSION_DIR}"

In [ ]:
# ON-DEVICE NOTE: IMU data arrives sample by sample in real time.
# Algorithms developed here should be written as streaming operations
# where possible, not as batch operations over the full session array.

import json
import pandas as pd

accel_rows, gyro_rows, mag_rows, baro_rows = [], [], [], []

with open(SESSION_DIR / "frames.ndjson") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        r = json.loads(line)
        fn = r["frame_number"]
        for s in r.get("accelerometer", []):
            accel_rows.append({**s, "frame_number": fn})
        for s in r.get("gyroscope", []):
            gyro_rows.append({**s, "frame_number": fn})
        for s in r.get("magnetometer", []):
            mag_rows.append({**s, "frame_number": fn})
        for s in r.get("barometer", []):
            baro_rows.append({**s, "frame_number": fn})

accel_df = pd.DataFrame(accel_rows)
gyro_df  = pd.DataFrame(gyro_rows)
mag_df   = pd.DataFrame(mag_rows)
baro_df  = pd.DataFrame(baro_rows)

# Convert ts_app to seconds from session start for readable x-axis
t0 = accel_df["ts_app"].min()
for df in [accel_df, gyro_df, mag_df, baro_df]:
    if not df.empty:
        df["t_sec"] = (df["ts_app"] - t0) / 1e9

print(f"Accelerometer samples : {len(accel_df)}")
print(f"Gyroscope samples     : {len(gyro_df)}")
print(f"Magnetometer samples  : {len(mag_df)}")
print(f"Barometer samples     : {len(baro_df)}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(accel_df["t_sec"], accel_df["x"], label="x", alpha=0.7, linewidth=0.8)
ax.plot(accel_df["t_sec"], accel_df["y"], label="y", alpha=0.7, linewidth=0.8)
ax.plot(accel_df["t_sec"], accel_df["z"], label="z", alpha=0.7, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("m/s\u00b2")
ax.set_title("Accelerometer")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(gyro_df["t_sec"], gyro_df["x"], label="x", alpha=0.7, linewidth=0.8)
ax.plot(gyro_df["t_sec"], gyro_df["y"], label="y", alpha=0.7, linewidth=0.8)
ax.plot(gyro_df["t_sec"], gyro_df["z"], label="z", alpha=0.7, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("rad/s")
ax.set_title("Gyroscope")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(mag_df["t_sec"], mag_df["x"], label="x", alpha=0.7, linewidth=0.8)
ax.plot(mag_df["t_sec"], mag_df["y"], label="y", alpha=0.7, linewidth=0.8)
ax.plot(mag_df["t_sec"], mag_df["z"], label="z", alpha=0.7, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("\u03bcT")
ax.set_title("Magnetometer")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
if not baro_df.empty:
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(baro_df["t_sec"], baro_df["pressure_hpa"], color="teal", linewidth=0.8)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("hPa")
    ax.set_title("Barometer")
    plt.tight_layout()
    plt.show()
else:
    print("No barometer data in this session")

In [ ]:
import json
with open(SESSION_DIR / "session_meta.json") as f:
    meta = json.load(f)

print("── Timestamp sources for this session ───────")
print(f"  imu_ts_source         : {meta.get('imu_ts_source')}")
print(f"  cross_sensor_alignment: {meta.get('cross_sensor_alignment')}")
print()
print("Use ts_sensor for physics calculations.")
print("Use ts_app for wall-clock alignment with audio.")